# morphological_quantification_2026-01-02 — 03_posterior_annotation

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 03 | Manual Posterior Annotation

Annotate one posterior-near click per organoid using the consensus axis from `02` and one representative retained z plane per file.


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define output paths.
- `Load Consensus Geometry Inputs`: load the consensus-axis table and merge it with the representative display-plane metadata from `02`.
- `Posterior Atlas Pages`: save image-first atlas pages with the consensus axis over each organoid before interactive annotation.
- `Posterior Progress`: summarize how many files already have saved posterior clicks.
- `Interactive Posterior Click Session`: click near the posterior end, save, and advance through the cohort.
- `Refresh Progress`: rerun after annotation to confirm saved coverage.
- `Next Step`: use the saved posterior orientation for marker-domain quantification and axis-aware measurements.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().resolve().parent
elif (Path.cwd() / "notebooks").exists():
    ROOT = Path.cwd().resolve()
else:
    raise RuntimeError("Run this notebook from the project root or the notebooks/ directory.")

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh


## Settings Notes

- This notebook uses the **consensus axis** exported by `02`, not a fresh geometry fit.
- Files flagged `include_in_analysis = False` in `results/manifests/analysis_manifest.tsv` are filtered out here.
- It shows one representative retained z plane per file, using the `display_z_index` stored in the consensus table.
- The posterior click is a lightweight orientation cue: click **near** the posterior end rather than worrying about the exact tip pixel.
- `Pass` keeps any existing saved click unchanged and advances. `Save Point + Next` writes the current click to disk.
- The widget starts at the first file without a saved posterior click when possible.


In [ ]:
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
CONSENSUS_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_consensus_geometry.tsv"
PER_Z_GEOMETRY_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_per_z_geometry.tsv"
POSTERIOR_PATH = ROOT / "results" / "annotations" / "manual_posterior_clicks.tsv"
POSTERIOR_ATLAS_QC_DIR = ROOT / "results" / "qc" / "posterior_annotation_atlas"

ATLAS_ROWS_PER_PAGE = 4
MAX_INLINE_ATLAS_PAGES = 2
REBUILD_POSTERIOR_ATLAS = False

POSTERIOR_ATLAS_QC_DIR.mkdir(parents=True, exist_ok=True)
POSTERIOR_PATH.parent.mkdir(parents=True, exist_ok=True)


## Load Consensus Geometry Inputs


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
included_file_paths = set(
    manifest_df.loc[
        manifest_df["include_in_analysis"].fillna(True).astype(bool),
        "file_path",
    ].astype(str)
)
excluded_markers_by_file_path = mqh.marker_exclusion_map_from_manifest(manifest_df)
excluded_marker_summary = [
    {
        "file_path": file_path,
        "excluded_marker_keys": ",".join(sorted(marker_keys)),
    }
    for file_path, marker_keys in sorted(excluded_markers_by_file_path.items())
]
excluded_marker_summary_df = pd.DataFrame(excluded_marker_summary)
consensus_df = pd.read_csv(CONSENSUS_TABLE_PATH, sep="\t")
consensus_df = consensus_df.loc[consensus_df["file_path"].astype(str).isin(included_file_paths)].copy()
per_z_geometry_df = pd.read_csv(PER_Z_GEOMETRY_TABLE_PATH, sep="\t")
per_z_geometry_df = per_z_geometry_df.loc[
    per_z_geometry_df["file_path"].astype(str).isin(included_file_paths)
].copy()
annotation_df = mqh.build_posterior_annotation_input_table(
    consensus_df=consensus_df,
    per_z_geometry_df=per_z_geometry_df,
)
records = mqh.records_from_manifest(annotation_df)
posterior_df = mqh.load_posterior_table(POSTERIOR_PATH)
posterior_df = posterior_df.loc[posterior_df["file_path"].astype(str).isin(included_file_paths)].copy()
annotation_preview_df = annotation_df.merge(
    posterior_df[["file_path", "posterior_click_x_px", "posterior_click_y_px"]]
    if not posterior_df.empty
    else pd.DataFrame(columns=["file_path", "posterior_click_x_px", "posterior_click_y_px"]),
    on="file_path",
    how="left",
)

print(f"Consensus annotation rows: {len(annotation_df)}")
print(f"Files with existing posterior clicks: {int(annotation_preview_df['posterior_click_x_px'].notna().sum())}")


## Posterior Atlas Pages


In [ ]:
atlas_pages = []
file_ids = sorted(annotation_preview_df["file_id"].unique())
existing_atlas_pages = sorted(POSTERIOR_ATLAS_QC_DIR.glob("posterior_annotation_atlas_page*.png"))

if not file_ids:
    print("No files were available for posterior annotation.")
elif existing_atlas_pages and not REBUILD_POSTERIOR_ATLAS:
    atlas_pages = existing_atlas_pages
    print(
        f"Reusing {len(atlas_pages)} existing atlas page(s) from {POSTERIOR_ATLAS_QC_DIR}. "
        "Set REBUILD_POSTERIOR_ATLAS = True to regenerate them."
    )
else:
    for page_idx, start_idx in enumerate(range(0, len(file_ids), ATLAS_ROWS_PER_PAGE), start=1):
        page_file_ids = file_ids[start_idx : start_idx + ATLAS_ROWS_PER_PAGE]
        n_rows = len(page_file_ids)
        fig, axes = plt.subplots(
            n_rows,
            2,
            figsize=(11.8, 4.2 * n_rows),
            gridspec_kw={"width_ratios": [1.0, 0.36]},
            constrained_layout=True,
        )
        axes = np.asarray(axes)
        if axes.ndim == 1:
            axes = axes[None, :]

        for row_idx, file_id in enumerate(page_file_ids):
            img_ax = axes[row_idx, 0]
            note_ax = axes[row_idx, 1]
            row = (
                annotation_preview_df[annotation_preview_df["file_id"] == int(file_id)]
                .sort_values("file_id")
                .iloc[0]
            )

            plane = mqh.load_plane_channels(
                ROOT / str(row.file_path),
                z_index=int(row.display_z_index),
            )
            display_mask = mqh.load_binary_mask(ROOT / str(row.display_mask_path))
            consensus_geometry = mqh.consensus_geometry_from_annotation_row(
                row=row,
                per_z_geometry_df=per_z_geometry_df,
                root=ROOT,
                average_mode="mean",
                n_points=121,
            )
            posterior_xy = None
            if np.isfinite(pd.to_numeric(row.posterior_click_x_px, errors="coerce")) and np.isfinite(
                pd.to_numeric(row.posterior_click_y_px, errors="coerce")
            ):
                posterior_xy = np.array(
                    [
                        float(pd.to_numeric(row.posterior_click_x_px)),
                        float(pd.to_numeric(row.posterior_click_y_px)),
                    ],
                    dtype=np.float64,
                )

            img_ax.imshow(plane["overlay_rgb"])
            mqh.plot_mask_outline(img_ax, display_mask, color="yellow", linewidth=1.0)
            mqh.plot_centerline_overlay(
                img_ax,
                centerline_xy=consensus_geometry["centerline_xy"],
                midpoint_xy=consensus_geometry["midpoint_xy"],
                endpoint_a_xy=consensus_geometry["endpoint_a_xy"],
                endpoint_b_xy=consensus_geometry["endpoint_b_xy"],
                base_endpoint_a_xy=None,
                base_endpoint_b_xy=None,
                posterior_click_xy=posterior_xy,
                line_color="white",
                line_width=2.2,
            )
            img_ax.set_title(
                f"{int(file_id):02d} | display z={int(row.display_z_index)} | kept {int(row.n_retained_z)}/{int(row.n_total_z)}",
                fontsize=10,
            )
            img_ax.set_xticks([])
            img_ax.set_yticks([])

            retained_text = str(row.retained_z_indices) if pd.notna(row.retained_z_indices) else str(int(row.display_z_index))
            omitted_text = str(row.omitted_z_indices) if pd.notna(row.omitted_z_indices) else "None"
            click_text = "yes" if posterior_xy is not None else "no"
            note_ax.axis("off")
            note_ax.text(
                0.02,
                0.92,
                (
                    f"retained z: {retained_text}\n"
                    f"omitted z: {omitted_text}\n"
                    f"saved click: {click_text}"
                ),
                transform=note_ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
            )

        atlas_page = POSTERIOR_ATLAS_QC_DIR / f"posterior_annotation_atlas_page{page_idx:02d}.png"
        fig.savefig(atlas_page, dpi=180, bbox_inches="tight")
        plt.close(fig)
        atlas_pages.append(atlas_page)

    print(f"Saved {len(atlas_pages)} atlas page(s) to {POSTERIOR_ATLAS_QC_DIR}")

for atlas_page in atlas_pages[:MAX_INLINE_ATLAS_PAGES]:
    print(atlas_page.relative_to(ROOT))
    display(Image(filename=str(atlas_page)))


## Posterior Progress


In [ ]:
progress_df = mqh.posterior_progress_for_records(records, POSTERIOR_PATH)
n_done = int(progress_df["has_posterior_click"].sum())
n_total = int(len(progress_df))
remaining_ids = progress_df.loc[~progress_df["has_posterior_click"], "file_id"].astype(int).tolist()

print(f"Posterior clicks saved: {n_done}/{n_total}")
if remaining_ids:
    preview = ", ".join(f"{v:02d}" for v in remaining_ids[:15])
    suffix = " ..." if len(remaining_ids) > 15 else ""
    print(f"Remaining file IDs: {preview}{suffix}")
else:
    print("All files already have posterior clicks saved.")


## Interactive Backend


In [ ]:
# Match the sibling manual-annotation notebooks: use the Matplotlib widget backend.
ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "widget")
    except Exception as exc:
        print("Could not enable '%matplotlib widget'. Interactive clicks may not work.")
        print("Install ipympl if needed: pip install ipympl")
        print("Details:", exc)


## Interactive Posterior Click Session


In [ ]:
posterior_session = mqh.ManualConsensusPosteriorPointSession(
    annotation_df=annotation_df,
    per_z_geometry_df=per_z_geometry_df,
    root=ROOT,
    posterior_path=POSTERIOR_PATH,
    allow_add=True,
)
posterior_session.render()


## Refresh Progress


In [ ]:
refreshed_progress_df = mqh.posterior_progress_for_records(records, POSTERIOR_PATH)
print(
    f"Posterior clicks saved after refresh: "
    f"{int(refreshed_progress_df['has_posterior_click'].sum())}/{int(len(refreshed_progress_df))}"
)


## Next Step

Use the saved posterior clicks in `results/annotations/manual_posterior_clicks.tsv` to orient the consensus axis before calling FOXF1, PAX8, and MESP2 domains in the next notebook.
